# BinWaves example in Cantabria (Validation)

**In this notebook**: 
<br><br>
Here waves are reconstructed at the buoy location for comparison.
<br><br>
Steps:
- Buoy is loaded.
- Kp propagation coefficients and hindcast reconstruction is made at the buoy location.
- Comparison plots and statistics are shown.

In [ ]:
import pandas as pd
import xarray as xr

# Load buoy data and kps

buoy_waves = pd.read_pickle("outputs/buoy_41025_bulk_parameters.pkl").sort_index().loc["2016"]
kp_coeffs = xr.open_dataset("outputs/kp_coefficients.nc").isel(site=[2])
kp_coeffs

In [ ]:
kp_coeffs.utm_x.values, kp_coeffs.utm_y.values

In [ ]:
from utils.operations import transform_ERA5_spectrum
import numpy as np
model_parameters = pd.read_csv("NC_cases/swan_cases.csv").to_dict(orient="list")

# Load interest spectra
offshore_spectra, offshore_spectra_case = (  # Unpack both values from the tuple
    transform_ERA5_spectrum(
        era5_spectrum=xr.open_dataset("outputs/jen_north_carolina_spec_utm_unique.nc"),
        subset_parameters=model_parameters,
        available_case_num=kp_coeffs.case_num.values,
    )
)

# Add coordinates directly to offshore_spectra
# # buoy 44088
# offshore_spectra_case.coords['longitude'] = np.float32(285.161)
# offshore_spectra_case.coords['latitude'] = np.float32(36.6120)
# buoy 44100
# offshore_spectra_case.coords['longitude'] = np.float32(284.407)
# offshore_spectra_case.coords['latitude'] = np.float32(36.2580)
# # buoy 44056
# offshore_spectra_case.coords['longitude'] = np.float32(284.286)
# offshore_spectra_case.coords['latitude'] = np.float32(36.2000)
# # buoy 41120
# offshore_spectra_case.coords['longitude'] = np.float32(284.715)
# offshore_spectra_case.coords['latitude'] = np.float32(35.2580)
# buoy 41025
offshore_spectra_case.coords['longitude'] = np.float32(284.546)
offshore_spectra_case.coords['latitude'] = np.float32(35.0100)
# # buoy 44095
# offshore_spectra_case.coords['longitude'] = np.float32(284.67)
# offshore_spectra_case.coords['latitude'] = np.float32(35.7500)
# # buoy 44086
# offshore_spectra_case.coords['longitude'] = np.float32(284.579)
# offshore_spectra_case.coords['latitude'] = np.float32(36.0010)

# Add attributes to the coordinates
offshore_spectra_case.coords['longitude'].attrs = {
    'units': 'degrees_east',
    'long_name': 'Longitude'
}
offshore_spectra_case.coords['latitude'].attrs = {
    'units': 'degrees_north',
    'long_name': 'Latitude'
}
offshore_spectra_case

In [22]:
# import xarray as xr

# # Load the dataset
# offshore_spectra_case = xr.open_dataset("outputs/jen_north_carolina_spec_utm_unique.nc")


# import numpy as np
# times = offshore_spectra_case.time.to_index()
# unique_times, unique_idx = np.unique(offshore_spectra_case.time.values, return_index=True)
# offshore_spectra_case = offshore_spectra_case.isel(time=unique_idx)
# offshore_spectra_case.to_netcdf("outputs/jen_north_carolina_spec_utm_unique.nc")

In [ ]:
from bluemath_tk.waves.binwaves import reconstruc_spectra
# Reconstruct spectra

reconstructed_onshore_spectra = reconstruc_spectra(
    offshore_spectra=offshore_spectra_case.sel(time=buoy_waves.index, method="nearest"),
    kp_coeffs=kp_coeffs,
)
reconstructed_onshore_spectra

In [ ]:
from utils.plotting import plot_wave_series

# Plot reconstructed bulk parameters vs buoy data

# plot_wave_series(
#     buoy_data=buoy_waves,
#     binwaves_data=reconstructed_onshore_spectra.rename({"kps": "efth"})
#     .squeeze()
#     .sel(time=buoy_waves.index, method="nearest")
#     .spec,
#     offshore_data=offshore_spectra.sel(time=buoy_waves.index, method="nearest").spec,
#     times=buoy_waves.index.values,
# )
# Plot reconstructed bulk parameters vs buoy data
plot_wave_series(
    buoy_data=buoy_waves,
    binwaves_data=reconstructed_onshore_spectra.rename({"kps": "efth"})
    .squeeze()
    .drop_duplicates('time')  # Add this line to remove duplicates
    .sel(time=buoy_waves.index, method="nearest")
    .spec,
    offshore_data=offshore_spectra.sel(time=buoy_waves.index, method="nearest").spec,
    times=buoy_waves.index.values,
)